In [1]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf

from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# Definindo os caminhos - notebook em scripts/Treinamento DNN/
caminho_pasta_tratado = '../../dataset tratado/lycos-cicids2017/'

nome_dados_treinamento = 'Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos_balanceados.csv'
nome_dados_teste = 'Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos_balanceados.csv'

In [3]:
print("Carregando os datasets com redução de dimensionalidade (balanceado)...")
df_treino = pd.read_csv(caminho_pasta_tratado + nome_dados_treinamento)
df_teste = pd.read_csv(caminho_pasta_tratado + nome_dados_teste)

print(f"Dados carregados! Treino: {df_treino.shape} | Teste: {df_teste.shape}")
display(df_treino.head())
display(df_teste.head())

Carregando os datasets com redução de dimensionalidade (balanceado)...
Dados carregados! Treino: (1286248, 52) | Teste: (551250, 52)


,src_port,dst_port,bwd_pkt_len_max,flow_duration,bwd_pkt_len_mean,pkt_len_max,fwd_pkt_len_max,down_up_ratio,fwd_pkt_hdr_len_tot,fwd_pkt_len_std,...,fwd_flag_psh,bwd_pkt_len_tot,bwd_iat_std,bwd_pkt_cnt,fwd_iat_min,flag_SYN,fwd_pkt_per_s,bwd_pkt_per_s,idle_min,Label
0,0.951110,0.000809,0.005018,0.000504,0.022422,0.003948,0.001853,0.123909,0.000002,0.000000,...,0.000000,1.495151e-07,0.000000,0.000003,0.000000e+00,0.000000,8.271162e-06,8.271162e-06,0.000000,benign
1,0.766598,0.006760,0.000000,0.025492,0.000000,0.001249,0.001249,0.041303,0.000013,0.002472,...,0.000629,0.000000e+00,0.000000,0.000003,4.000014e-07,0.000000,4.903540e-07,1.634515e-07,0.000000,benign
2,0.814466,0.006760,0.072606,0.979619,0.061572,0.057131,0.023852,0.111518,0.000142,0.019246,...,0.005035,7.390316e-06,0.233425,0.000062,8.333363e-09,0.038462,8.506700e-08,7.656050e-08,0.485929,benign
3,0.074983,0.000809,0.013313,0.000002,0.059487,0.010475,0.001330,0.123909,0.000003,0.000000,...,0.000000,7.933453e-07,0.000000,0.000007,2.500009e-08,0.000000,3.875969e-03,3.875969e-03,0.000000,benign
4,0.847898,0.006760,0.222427,0.070290,0.387286,0.175020,0.020830,0.149890,0.000462,0.014427,...,0.003147,1.936876e-04,0.007837,0.000257,8.333363e-09,0.038462,3.675254e-06,4.445871e-06,0.000000,benign


,src_port,dst_port,bwd_pkt_len_max,flow_duration,bwd_pkt_len_mean,pkt_len_max,fwd_pkt_len_max,down_up_ratio,fwd_pkt_hdr_len_tot,fwd_pkt_len_std,...,fwd_flag_psh,bwd_pkt_len_tot,bwd_iat_std,bwd_pkt_cnt,fwd_iat_min,flag_SYN,fwd_pkt_per_s,bwd_pkt_per_s,idle_min,Label
0,0.918364,0.000809,0.009063,1.508333e-06,0.040497,0.007131,0.001491,0.123909,0.000003,0.0,...,0.0,5.400850e-07,0.0,0.000007,2.500009e-08,0.000000,0.005525,0.005525,0.0,benign
1,0.838499,0.000809,0.004864,1.383333e-06,0.021736,0.003828,0.001894,0.123909,0.000003,0.0,...,0.0,2.898762e-07,0.0,0.000007,2.500009e-08,0.000000,0.006024,0.006024,0.0,benign
2,0.748730,0.000809,0.010753,2.175000e-06,0.048047,0.008461,0.001531,0.123909,0.000003,0.0,...,0.0,6.407789e-07,0.0,0.000007,4.083348e-07,0.000000,0.003831,0.003831,0.0,benign
3,0.556161,0.017473,0.000000,5.583333e-07,0.000000,0.000000,0.000000,0.123909,0.000005,0.0,...,0.0,0.000000e+00,0.0,0.000003,0.000000e+00,0.019231,0.007463,0.007463,0.0,portscan
4,0.754528,0.001511,0.000000,1.000000e-06,0.000000,0.000000,0.000000,0.123909,0.000009,0.0,...,0.0,0.000000e+00,0.0,0.000003,0.000000e+00,0.019231,0.004167,0.004167,0.0,portscan


In [4]:
X_treino = df_treino.drop('Label', axis=1)
y_treino = df_treino['Label']

X_teste = df_teste.drop('Label', axis=1)
y_teste = df_teste['Label']

In [5]:
print("Codificando as labels para a Rede Neural...")
encoder = LabelEncoder()
y_treino_num = encoder.fit_transform(y_treino)
y_teste_num = encoder.transform(y_teste)
num_classes = len(encoder.classes_)

Codificando as labels para a Rede Neural...


In [6]:
dnn_model = Sequential([
    keras.layers.Input(shape=(X_treino.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

dnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

In [7]:
print("\nIniciando o treinamento da Rede Neural Profunda (DNN) com dimensionalidade reduzida (balanceado)...")
inicio_treino_dnn = time.time()

history = dnn_model.fit(X_treino, y_treino_num, epochs=10, batch_size=256, validation_split=0.1)

fim_treino_dnn = time.time()
tempo_treinamento_dnn = fim_treino_dnn - inicio_treino_dnn
print(f"\nTreinamento DNN concluído! Tempo total: {tempo_treinamento_dnn:.4f} segundos.")


Iniciando o treinamento da Rede Neural Profunda (DNN) com dimensionalidade reduzida (balanceado)...
Epoch 1/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9831 - loss: 0.0729 - val_accuracy: 0.9951 - val_loss: 0.0177
Epoch 2/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.9954 - loss: 0.0169 - val_accuracy: 0.9967 - val_loss: 0.0109
Epoch 3/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.9964 - loss: 0.0122 - val_accuracy: 0.9970 - val_loss: 0.0093
Epoch 4/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.9968 - loss: 0.0105 - val_accuracy: 0.9971 - val_loss: 0.0095
Epoch 5/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.9970 - loss: 0.0097 - val_accuracy: 0.9971 - val_loss: 0.0085
Epoch 6/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.9971 - loss: 0.0089 - val_accuracy: 0.9975 - val_loss: 0.0076
Epoch 7/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9972 - loss: 0.0085 - val_accuracy: 0.9975 - val_lo

In [8]:
print("\nIniciando a classificação dos dados de teste...")
inicio_teste_dnn = time.time()

previsoes_probabilidades = dnn_model.predict(X_teste)
previsoes_dnn_num = np.argmax(previsoes_probabilidades, axis=1)

fim_teste_dnn = time.time()
tempo_teste_dnn = fim_teste_dnn - inicio_teste_dnn
print(f"Classificação concluída! Tempo de previsão: {tempo_teste_dnn:.4f} segundos.")

previsoes_dnn_labels = encoder.inverse_transform(previsoes_dnn_num)

print("\n=== MATRIZ DE CONFUSÃO (DNN - Reduzida, Balanceada) ===")
labels = sorted(np.unique(np.concatenate([y_teste.values, previsoes_dnn_labels])))
cm = confusion_matrix(y_teste, previsoes_dnn_labels, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Real_{lbl}" for lbl in labels],
    columns=[f"Pred_{lbl}" for lbl in labels]
)

def color_confusion_matrix(data):
    styles = pd.DataFrame("", index=data.index, columns=data.columns)
    for i in range(min(data.shape[0], data.shape[1])):
        styles.iat[i, i] = "background-color: #c6efce; color: #006100; font-weight: bold;"
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if i != j and data.iat[i, j] != 0:
                styles.iat[i, j] = "background-color: #ffc7ce; color: #9c0006; font-weight: bold;"
    return styles

display(cm_df.style.apply(color_confusion_matrix, axis=None).format("{:.0f}"))

print("\n=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===")
print(classification_report(y_teste, previsoes_dnn_labels, zero_division=0))


Iniciando a classificação dos dados de teste...
17227/17227 ━━━━━━━━━━━━━━━━━━━━ 8s 446us/step
Classificação concluída! Tempo de previsão: 11.9104 segundos.

=== MATRIZ DE CONFUSÃO (DNN - Reduzida, Balanceada) ===


,Pred_benign,Pred_bot,Pred_ddos,Pred_dos_goldeneye,Pred_dos_hulk,Pred_dos_slowhttptest,Pred_dos_slowloris,Pred_ftp_patator,Pred_heartbleed,Pred_portscan,Pred_ssh_patator,Pred_webattack_bruteforce,Pred_webattack_sql_injection,Pred_webattack_xss
Real_benign,418250,6,1,255,3,84,14,1,0,24,1,0,0,0
Real_bot,0,219,0,0,0,0,0,0,0,0,0,0,0,0
Real_ddos,9,0,28587,0,0,0,0,0,0,0,0,0,0,0
Real_dos_goldeneye,4,0,0,2090,1,1,0,0,0,0,0,0,0,0
Real_dos_hulk,5,0,0,0,47934,0,0,0,0,3,0,0,0,0
Real_dos_slowhttptest,40,0,0,1,0,1379,3,0,0,0,0,0,0,0
Real_dos_slowloris,52,0,0,0,0,11,1703,0,0,9,0,0,0,0
Real_ftp_patator,3,0,0,0,0,0,0,1178,0,0,0,0,0,0
Real_heartbleed,0,0,0,0,0,0,0,0,4,0,0,0,0,0
Real_portscan,17,0,0,0,0,0,1,0,0,47889,0,0,0,0



=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===
                         precision    recall  f1-score   support

                 benign       1.00      1.00      1.00    418639
                    bot       0.97      1.00      0.99       219
                   ddos       1.00      1.00      1.00     28596
          dos_goldeneye       0.89      1.00      0.94      2096
               dos_hulk       1.00      1.00      1.00     47942
       dos_slowhttptest       0.93      0.97      0.95      1423
          dos_slowloris       0.99      0.96      0.97      1775
            ftp_patator       1.00      1.00      1.00      1181
             heartbleed       1.00      1.00      1.00         4
               portscan       1.00      1.00      1.00     47907
            ssh_patator       1.00      0.98      0.99       875
   webattack_bruteforce       0.96      0.07      0.12       397
webattack_sql_injection       0.00      0.00      0.00         3
          webattack_xss      

In [9]:
CLASS_ALIASES_LATEX = {'benign': 'BENIGN', 'bot': 'Bot', 'ddos': 'DDoS', 'dos_goldeneye': 'DoS GE', 'dos_hulk': 'Hulk', 'dos_slowhttptest': 'Slowhttp', 'dos_slowloris': 'Slowloris', 'ftp_patator': 'FTP', 'heartbleed': 'Heart', 'portscan': 'PortScan', 'ssh_patator': 'SSH', 'webattack_bruteforce': 'Brute', 'webattack_sql_injection': 'SQL', 'webattack_xss': 'XSS'}


def escape_latex(value):
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in str(value))


def latex_class_name(label, short=False):
    if short:
        return escape_latex(CLASS_ALIASES_LATEX.get(label, label))
    return escape_latex(label)


def format_confusion_value(value, is_diagonal):
    value = int(value)
    if is_diagonal:
        return rf"\ok{{{value}}}"
    if value != 0:
        return rf"\err{{{value}}}"
    return "0"


def make_latex_confusion_matrix(cm_values, class_labels, caption, table_label):
    headers = [latex_class_name(label, short=True) for label in class_labels]
    rows = []
    for i, real_label in enumerate(class_labels):
        row_values = [format_confusion_value(cm_values[i][j], i == j) for j in range(len(class_labels))]
        rows.append((rf"Real\_{latex_class_name(real_label, short=True)}", row_values))

    first_col_width = max([0] + [len(row_name) for row_name, _ in rows])
    col_widths = [max(len(headers[i]), *(len(values[i]) for _, values in rows)) for i in range(len(headers))]

    def format_row(first_cell, values):
        first = first_cell.ljust(first_col_width)
        rest = " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values))
        return f"            {first} & {rest} \\\\"

    lines = [
        r"\begin{table}[H]",
        r"    \centering",
        r"    \scriptsize",
        r"    \resizebox{\linewidth}{!}{",
        rf"        \begin{{tabular}}{{l|{'r' * len(class_labels)}}}",
        r"            \hline",
        format_row("", headers),
        r"            \hline",
    ]
    lines.extend(format_row(row_name, row_values) for row_name, row_values in rows)
    lines.extend([
        r"            \hline",
        r"        \end{tabular}",
        r"    }",
        rf"    \caption{{{escape_latex(caption)}}}",
        rf"    \label{{{table_label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)


def format_metric(value):
    return "-" if value is None else f"{value:.2f}"


def make_latex_metrics_report(y_true_values, y_pred_values, class_labels, caption, table_label):
    report = classification_report(
        y_true_values,
        y_pred_values,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )
    total_support = int(sum(report[label]["support"] for label in class_labels))
    rows = []
    for label in class_labels:
        metrics = report[label]
        rows.append([
            latex_class_name(label),
            format_metric(metrics["precision"]),
            format_metric(metrics["recall"]),
            format_metric(metrics["f1-score"]),
            str(int(metrics["support"])),
        ])

    rows.extend([
        [r"\textbf{Acurácia}", "-", format_metric(report["accuracy"]), "-", str(total_support)],
        [r"\textbf{Média Macro}", format_metric(report["macro avg"]["precision"]), format_metric(report["macro avg"]["recall"]), format_metric(report["macro avg"]["f1-score"]), str(total_support)],
        [r"\textbf{Média Ponderada}", format_metric(report["weighted avg"]["precision"]), format_metric(report["weighted avg"]["recall"]), format_metric(report["weighted avg"]["f1-score"]), str(total_support)],
    ])

    headers = ["Classe", "Precisão", "Revocação", "F1-score", "Suporte"]
    col_widths = [max(len(str(row[i])) for row in [headers] + rows) for i in range(len(headers))]

    def format_row(values):
        return "        " + " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values)) + r" \\"

    lines = [
        r"\begin{table}[H]",
        r"    \centering",
        r"    \small",
        r"    \begin{tabular}{lrrrr}",
        r"        \hline",
        format_row(headers),
        r"        \hline",
    ]
    lines.extend(format_row(row) for row in rows[:len(class_labels)])
    lines.extend([
        r"        \hline",
        format_row(rows[-3]),
        format_row(rows[-2]),
        format_row(rows[-1]),
        r"        \hline",
        r"    \end{tabular}",
        rf"    \caption{{{escape_latex(caption)}}}",
        rf"    \label{{{table_label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)


labels_latex = labels
y_true_latex = y_teste
y_pred_latex = previsoes_dnn_labels

latex_matriz_confusao = make_latex_confusion_matrix(
    cm,
    labels_latex,
    "Matriz de Confusão — DNN com Redução de Dimensionalidade Balanceada (LycoS-IDS2017)",
    "table:dnn_lycos_reducao_balanced_mc",
)
latex_relatorio_metricas = make_latex_metrics_report(
    y_true_latex,
    y_pred_latex,
    labels_latex,
    "Relatório de Métricas — DNN com Redução de Dimensionalidade Balanceada (LycoS-IDS2017)",
    "table:dnn_lycos_reducao_balanced_metricas",
)

print(latex_matriz_confusao)
print("\n")
print(latex_relatorio_metricas)

\begin{table}[H]
    \centering
    \scriptsize
    \resizebox{\linewidth}{!}{
        \begin{tabular}{l|rrrrrrrrrrrrrr}
            \hline
                            & BENIGN      & Bot      & DDoS       & DoS GE    & Hulk       & Slowhttp  & Slowloris & FTP       & Heart  & PortScan   & SSH      & Brute   & SQL    & XSS    \\
            \hline
            Real\_BENIGN    & \ok{418250} & \err{6}  & \err{1}    & \err{255} & \err{3}    & \err{84}  & \err{14}  & \err{1}   & 0      & \err{24}   & \err{1}  & 0       & 0      & 0      \\
            Real\_Bot       & 0           & \ok{219} & 0          & 0         & 0          & 0         & 0         & 0         & 0      & 0          & 0        & 0       & 0      & 0      \\
            Real\_DDoS      & \err{9}     & 0        & \ok{28587} & 0         & 0          & 0         & 0         & 0         & 0      & 0          & 0        & 0       & 0      & 0      \\
            Real\_DoS GE    & \err{4}     & 0        & 0          & \ok{2090}